In [2]:
import os
import ast
import json
import pickle
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELS     = CONFIGS['experiments']
SPLIT      = 'test'

SR_GAUSS_FIELDVARS = MODELS['sr']['runs']['sr_gauss']['fieldvars']
NN_SEEDS           = MODELS['nn']['seeds']
OPTIMIZEDEQS       = MODELS['sr']['optimizedeqs']
PODRUNS            = MODELS['pod']['runs']
NNRUNS             = MODELS['nn']['runs']
ORDER              = ['pod_bl','nn_bl','nn_full','nn_nonparam','nn_gauss','sr_lo','sr_bl','sr_med','sr_hi']

COLORS = {}
LABELS = {}
for name,cfg in {**PODRUNS,**NNRUNS,**OPTIMIZEDEQS}.items():
    COLORS[name] = cfg['color']
    LABELS[name] = cfg['description']

NBINS   = 20
MINSAMP = 30

In [ ]:
def kernel_integrate(fields,weights,dsig,mask=None):
    w = fields*weights[None,:,:]*dsig[None,None,:]
    if mask is not None:
        w = w*mask[:,None,:]
    return w.sum(axis=2)

def to_phys(srout):
    return np.expm1(tpstd*np.maximum(0.0,np.asarray(srout,dtype=float)))

SRFUNCTIONS = {
    'cube':lambda x:x**3,
    'square':lambda x:x**2,
    'neg':lambda x:-x,
    'sqrt':np.sqrt,
    'exp':np.exp,
    'log':np.log,
    'abs':np.abs,
    'max':np.maximum,
    'min':np.minimum}

def eval_form(form,variables,constants):
    ns = dict(SRFUNCTIONS)
    ns.update(variables)
    ns.update(constants)
    return np.asarray(eval(form,{'__builtins__':{}},ns),dtype=float)

def used_predictors(form,candidates):
    names = {n.id for n in ast.walk(ast.parse(form,mode='eval')) if isinstance(n,ast.Name)}
    return [c for c in candidates if c in names]

def bin1d(x,z,nbins=NBINS,minsamp=MINSAMP,plo=1,phi=99):
    '''Binned mean of z vs x using evenly-spaced value bins (not quantile bins).'''
    finite = np.isfinite(x)&np.isfinite(z)
    x,z    = x[finite],z[finite]
    lo,hi  = np.percentile(x,[plo,phi])
    edges  = np.linspace(lo,hi,nbins+1)
    xi     = np.clip(np.digitize(x,edges)-1,0,nbins-1)
    means  = np.full(nbins,np.nan)
    for i in range(nbins):
        sel = xi==i
        if sel.sum()>=minsamp:
            means[i] = z[sel].mean()
    return 0.5*(edges[:-1]+edges[1:]),means

def bin2d(x,y,z,nbins=NBINS,minsamp=MINSAMP,plo=1,phi=99):
    '''Binned mean of z vs (x,y) using evenly-spaced 2D value bins (not quantile bins).'''
    finite  = np.isfinite(x)&np.isfinite(y)&np.isfinite(z)
    x,y,z   = x[finite],y[finite],z[finite]
    xlo,xhi = np.percentile(x,[plo,phi])
    ylo,yhi = np.percentile(y,[plo,phi])
    xedges  = np.linspace(xlo,xhi,nbins+1)
    yedges  = np.linspace(ylo,yhi,nbins+1)
    xi      = np.clip(np.digitize(x,xedges)-1,0,nbins-1)
    yi      = np.clip(np.digitize(y,yedges)-1,0,nbins-1)
    flatidx = xi*nbins+yi
    sums    = np.bincount(flatidx,weights=z,minlength=nbins*nbins).reshape(nbins,nbins)
    counts  = np.bincount(flatidx,minlength=nbins*nbins).reshape(nbins,nbins)
    means   = np.where(counts>=minsamp,sums/np.maximum(counts,1),np.nan)
    return 0.5*(xedges[:-1]+xedges[1:]),0.5*(yedges[:-1]+yedges[1:]),means,counts

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
tpmean = float(STATS['tp_mean'])
tpstd  = float(STATS['tp_std'])
zmin   = (0.0-tpmean)/tpstd

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = ds.sizes['time'],ds.sizes['lat'],ds.sizes['lon']
    nsig            = ds.sizes.get('sig',1)
    dsig            = ds['dsig'].values
    farrs      = [ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in SR_GAUSS_FIELDVARS]
    fieldstack = np.stack(farrs,axis=1)
    surfmask   = (ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig)
                  if 'surfmask' in ds else None)
    def getflat(da):
        if 'time' in da.dims:
            return da.transpose('time','lat','lon').values.ravel()
        return np.tile(da.values,(ntime,1,1)).ravel()
    blnorm  = getflat(ds['bl'])
    lfnorm  = getflat(ds['lf'])
    shfnorm = getflat(ds['shf'])
    lhfnorm = getflat(ds['lhf'])

kwlist = []
for seed in NN_SEEDS:
    wpath = os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc')
    if os.path.exists(wpath):
        with xr.open_dataset(wpath,engine='h5netcdf') as wds:
            kwlist.append(wds['k'].values)

ki = np.mean([kernel_integrate(fieldstack,kw,dsig,surfmask) for kw in kwlist],axis=0) if kwlist else fieldstack.mean(axis=2)
rhk,thetaek,thetaestark = ki[:,0],ki[:,1],ki[:,2]

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsflat = ds.tp.transpose('time','lat','lon').values.ravel()

with open(os.path.join(MODELSDIR,'sr','optimized_equations.pkl'),'rb') as f:
    SR_REGISTRY = pickle.load(f)

In [ ]:
VARS = {'bl':blnorm,'rh':rhk,'thetae':thetaek,'thetaestar':thetaestark,'lf':lfnorm,'shf':shfnorm,'lhf':lhfnorm}

MODELPRED = {}
MODELPREDICTORS = {}
for name,cfg in OPTIMIZEDEQS.items():
    entry          = SR_REGISTRY.get(name,{})
    form           = entry.get('form',cfg['form'])
    constants      = entry.get('constants',cfg['init'])
    predictornames = used_predictors(form,VARS.keys())
    variables      = {p:VARS[p] for p in predictornames}
    formvals       = eval_form(form,variables,constants)
    MODELPRED[name]       = to_phys(formvals)
    MODELPREDICTORS[name] = predictornames

In [ ]:
def load_predictions(name,predsdir,split=SPLIT):
    '''Load saved POD/NN predictions (averaged across seeds if present), flattened to match obsflat.'''
    filepath = os.path.join(predsdir,f'{name}_{split}_predictions.nc')
    if not os.path.exists(filepath):
        return None
    with xr.open_dataset(filepath,engine='h5netcdf') as ds:
        da = ds['tp'].load()
    if 'seed' in da.dims:
        da = da.mean('seed')
    return da.transpose('time','lat','lon').values.ravel()

for name,cfg in PODRUNS.items():
    pred = load_predictions(name,PREDSDIR)
    if pred is None:
        continue
    MODELPRED[name]       = pred
    MODELPREDICTORS[name] = [cfg['inputvar']]

for name,cfg in NNRUNS.items():
    pred = load_predictions(name,PREDSDIR)
    if pred is None:
        continue
    MODELPRED[name]       = pred
    MODELPREDICTORS[name] = cfg['fieldvars']+cfg.get('localvars',[])

print(f'Loaded predictions for: {sorted(MODELPRED.keys())}')

In [ ]:
PHYSUNITS  = {'bl':'m/s$^2$','rh':'%','thetae':'K','thetaestar':'K','lf':'0-1','shf':'W/m$^2$','lhf':'W/m$^2$'}
PHYSLABELS = {'bl':'BL','rh':'RH','thetae':r'$\theta_e$','thetaestar':r'$\theta_e^*$','lf':'LF','shf':'SHF','lhf':'LHF'}

PREDICTORS = []
for p in ['bl','rh','thetae','thetaestar','lf','shf','lhf']:
    models = [name for name in MODELPREDICTORS if name in MODELPRED and p in MODELPREDICTORS[name]]
    if models:
        PREDICTORS.append((p,models))

def models_for_pair(px,py):
    return [name for name in MODELPREDICTORS
            if name in MODELPRED and px in MODELPREDICTORS[name] and py in MODELPREDICTORS[name]]

valid = np.isfinite(obsflat)
for arr in VARS.values():
    valid &= np.isfinite(arr)
for arr in MODELPRED.values():
    valid &= np.isfinite(arr)

print(f'Valid samples: {valid.sum():,}')
print(PREDICTORS)

In [ ]:
def plot_dependence(savepath):
    ncols = 3
    nrows = -(-len(PREDICTORS)//ncols)
    fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,figwidth=6.5,sharex=False,sharey=True)
    for ax,(p,models) in zip(axs,PREDICTORS):
        x = VARS[p][valid]*STATS[f'{p}_std']+STATS[f'{p}_mean']
        xc,obsbin = bin1d(x,obsflat[valid])
        ax.plot(xc,obsbin,color='k',linewidth=2,label='Observed',zorder=5)
        for name in ORDER:
            if name not in models:
                continue
            _,predbin = bin1d(x,MODELPRED[name][valid])
            ax.plot(xc,predbin,color=COLORS[name],linewidth=1.5,label=LABELS[name])
        ax.format(grid=False,xlabel=f'{PHYSLABELS[p]} ({PHYSUNITS[p]})')
    for ax in axs[len(PREDICTORS):]:
        ax.set_visible(False)
    axs[:,0].format(ylabel='Precipitation (mm)',ylim=(0,5))
    handles = [Line2D([],[],color='k',linewidth=2,label='Observed')]
    handles += [Line2D([],[],color=COLORS[name],linewidth=1.5,label=LABELS[name]) for name in ORDER if name in MODELPRED]
    fig.legend(handles,loc='r',ncols=1)
    # fig.format(abc=True,abcloc='ul')
    pplt.show()
    fig.save(savepath)
    return fig

In [ ]:
fig = plot_dependence(savepath='../figs/dependence_denormalized.jpg')

In [ ]:
def plot_dependence2d(px,py,savepath):
    '''2D partial dependence: binned-mean precipitation over (px,py), one panel per model that uses both.'''
    models = [name for name in ORDER if name in models_for_pair(px,py)]
    panels = ['obs']+models
    x = VARS[px][valid]*STATS[f'{px}_std']+STATS[f'{px}_mean']
    y = VARS[py][valid]*STATS[f'{py}_std']+STATS[f'{py}_mean']
    fig,axs = pplt.subplots(nrows=1,ncols=len(panels),figwidth=1.6*len(panels),share=False)
    vmax = None
    for ax,name in zip(axs,panels):
        z = obsflat[valid] if name=='obs' else MODELPRED[name][valid]
        xc,yc,zbin2d,_ = bin2d(x,y,z)
        if vmax is None:
            vmax = np.nanpercentile(zbin2d,99)
        m = ax.pcolormesh(xc,yc,zbin2d.T,cmap='DryWet',vmin=0,vmax=vmax,levels=21)
        title = 'Observed' if name=='obs' else LABELS[name]
        ax.format(title=title,xlabel=f'{PHYSLABELS[px]} ({PHYSUNITS[px]})',grid=False)
    axs[0].format(ylabel=f'{PHYSLABELS[py]} ({PHYSUNITS[py]})')
    fig.colorbar(m,loc='r',label='Precipitation (mm)')
    pplt.show()
    fig.save(savepath)
    return fig

In [ ]:
fig2d_thetae = plot_dependence2d('thetae','thetaestar',savepath='../figs/dependence2d_thetae_thetaestar.jpg')

In [ ]:
fig2d_flux = plot_dependence2d('lhf','shf',savepath='../figs/dependence2d_lhf_shf.jpg')